In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# ===========================
# Potential function classes
# ===========================
class LJ:
    def __init__(self, epsilon=1.0, sigma=1.0):
        self.epsilon = epsilon
        self.sigma = sigma
    def __call__(self, r):
        return 4.0 * self.epsilon * ((self.sigma / r) ** 12 - (self.sigma / r) ** 6)


class PatchyLJ:
    def __init__(self, epsilon=1.0, sigma=1.0, omega=0.262, patches=None):
        self.epsilon = epsilon
        self.sigma = sigma
        self.omega = omega
        self.LJ_model = LJ(epsilon, sigma)
        self.patches = patches if patches is not None else [0, 1.5708, 3.14159, 4.71239]

    def wrap(self, ang):
        return (ang + np.pi) % (2 * np.pi) - np.pi

    def __call__(self, r, phi_i, phi_j):
        pot = 0
        for patch_i in self.patches:
            for patch_j in self.patches:
                theta_i = self.wrap(phi_i + patch_i)
                theta_j = self.wrap(phi_j + patch_j - np.pi)
                pot += self.LJ_model(r) * np.exp(-(theta_i**2 + theta_j**2) / (2 * self.omega**2))
        return pot


class OrientedLJ:
    def __init__(self, epsilon=1, sigma=1, m=1, n=2, alpha=np.pi):
        self.m = m
        self.n = n
        self.alpha = alpha
        self.LJ_model = LJ(epsilon, sigma)
        
    def __call__(self, r, phi1, phi2):
        radial = self.LJ_model(r)
        angular = 1 + self.m * np.cos(self.n * (phi2 - phi1) + self.alpha)
        return radial + angular


class MorseWithAngles:
    def __init__(self, De=1.0, re=8.5, a=0.5, C1=0.3, C2=0.1):
        self.De = De
        self.re = re
        self.a = a
        self.C1 = C1
        self.C2 = C2

    def __call__(self, r, phi1, phi2):
        radial = self.De * (np.exp(-2 * self.a * (r - self.re)) - 2 * np.exp(-self.a * (r - self.re)))
        angular = 1 + self.C1 * np.cos(np.radians(phi2 - phi1)) \
                    + self.C2 * np.cos(2 * np.radians(phi2 - phi1))
        return radial * angular


class GeometricLJ:
    def __init__(self, epsilon=1, sigma=1, pathcNum=3, rho=1, factor=0.5, S_h=1):
        self.patchNum = pathcNum
        self.rho = rho
        self.factor = factor
        self.S_h = S_h
        self.LJ_model = LJ(epsilon, sigma)
        
    def __call__(self, r, phi1, phi2):
        dx = r
        dy = 0
        total_potential = self.LJ_model(r)
        for i in range(self.patchNum):
            theta1 = phi1 + self.S_h* 2*np.pi*i/self.patchNum
            theta2 = phi2 + 2*np.pi*i/self.patchNum
            
            delta_cos = self.rho * (np.cos(theta2) - np.cos(theta1))
            delta_sin = self.rho * (np.sin(theta2) - np.sin(theta1))
            
            r_patch = np.sqrt((dx + delta_cos) * (dx + delta_cos) + (dy + delta_sin) * (dy + delta_sin))
            total_potential += self.factor * self.LJ_model(r_patch)
            
        return total_potential
    
    
# ===========================
#       Plotting helper
# ===========================

def plot_potential(potential, r_fixed=9.0, n_points=100):
    phi = np.linspace(0, 2*np.pi, n_points)
    phi1 = np.linspace(0, 2*np.pi, n_points)
    phi2 = np.linspace(0, 2*np.pi, n_points)
    phi1_grid, phi2_grid = np.meshgrid(phi1, phi2)
    
    r_vals = np.linspace(0.9, 3, n_points)

    # 1D plots
    pot_1d_phi = [potential(r_fixed, p, 0) for p in phi]
    pot_1d_r = [potential(r, 0, 0) for r in r_vals]
    fig, ax1 = plt.subplots(nrows=1, ncols=2, figsize=(10, 3))
    ax1[0].plot(phi, pot_1d_phi)
    ax1[0].set_xlabel("phi_1")
    ax1[0].set_ylabel("Potential")
    ax1[0].grid(True)
    
    ax1[1].plot(r_vals, pot_1d_r)
    ax1[1].set_xlabel("r")
    ax1[1].set_ylabel("Potential")
    ax1[1].grid(True)

    # 2D plot
    pot_2d = np.array([[potential(r_fixed, p1, p2) for p1, p2 in zip(row1, row2)]
                       for row1, row2 in zip(phi1_grid, phi2_grid)])
    fig, ax2 = plt.subplots(figsize=(6, 5))
    im = ax2.imshow(pot_2d, origin='lower', extent=[0, 2*np.pi, 0, 2*np.pi], aspect='auto')
    ax2.set_xlabel("phi_1")
    ax2.set_ylabel("phi_2")
    fig.colorbar(im, ax=ax2, label="Potential")

    plt.show()

plt.rcParams.update({
    'font.size': 9, 'axes.labelsize': 10,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'figure.dpi': 150,
    'savefig.dpi': 600, 'text.usetex': False, 'mathtext.default': 'regular'
})

In [ ]:

pot = MorseWithAngles(De=1.0, re=8.5, a=0.5, C1=0.3, C2=0.1)
pot = OrientedLJ()
pot = PatchyLJ(epsilon=1.0, sigma=1.0, patches=[0, 2*np.pi/3, 4*np.pi/3])
pot = GeometricLJ(pathcNum=4, S_h=1)

plot_potential(pot, r_fixed=7)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

CONFIG = {
    'energy_ref': -6227.1749,
    'zeta_max': 1.39687500,
    'phi2_max': 20  # base range
}

def process_data(filename):
    df = pd.read_csv(filename, sep=r'\s+', header=None,
                     names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    df['energy'] -= 2 * CONFIG['energy_ref']
    df['zeta'] = np.where(
        df['zeta'] < CONFIG['zeta_max'] / 2,
        df['zeta'],
        df['zeta'] - CONFIG['zeta_max']
    )
    return df

def screw_pbc(df, screw_direction=20):
    """Repeat phi2 blocks to cover full 0–360 range with screw PBC."""
    max_phi1 = df['phi1'].max() + 1  # should be 360
    max_phi2 = CONFIG['phi2_max']    # should be 20
    repeats = 360 // max_phi2
    all_blocks = []

    for k in range(repeats):
        # Shift phi2 range
        temp = df.copy()
        temp['phi2'] = df['phi2'] +  k * max_phi2
        
        # Screw shift: roll phi1 by k * screw_direction * step
        temp['phi1'] = (-df['phi1'] + k * screw_direction) % max_phi1
        all_blocks.append(temp)

    return pd.concat(all_blocks, ignore_index=True)

def get_min_energy_per_phi(df):
    return df.loc[df.groupby(['phi1', 'phi2'])['energy'].idxmin()].reset_index(drop=True)

# --- Process and extend data ---
SSdata = process_data(data_path)
SSdata = screw_pbc(get_min_energy_per_phi(SSdata))
# SSdata_full = screw_pbc(SSdata, screw_direction=1)

# --- Pivot for plotting ---
grid_df = SSdata.pivot_table(index='phi2', columns='phi1', values='energy')
Phi1, Phi2 = np.meshgrid(grid_df.columns, grid_df.index)
E_grid = grid_df.values

# --- Plot ---
fig, ax = plt.subplots(figsize=(6,4))
c0 = ax.contourf(Phi1, Phi2, E_grid, levels=100, cmap='viridis')
ax.set_title("DFTB Energy Map")
ax.set_xlabel("phi1 (deg)")
ax.set_ylabel("phi2 (deg)")
fig.colorbar(c0, ax=ax, shrink=1)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import os

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

plt.rcParams.update({
    'font.size': 9, 'axes.labelsize': 10,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'figure.dpi': 150,
    'savefig.dpi': 600, 'text.usetex': False, 'mathtext.default': 'regular'
})

CONFIG = {
    'energy_ref': -6227.1749,
    'zeta_max': 1.39687500,
    'phi2_max': 20
}

# ------------------------
# Load and preprocess data
# ------------------------
def process_data(filename):
    df = pd.read_csv(filename, sep=r'\s+', header=None,
                     names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    df['energy'] -= 2 * CONFIG['energy_ref']
    df['zeta'] = np.where(df['zeta'] < CONFIG['zeta_max']/2,
                          df['zeta'], df['zeta'] - CONFIG['zeta_max'])
    df['phi2'] = (df['phi2'] + CONFIG['phi2_max']) % (2 * CONFIG['phi2_max']) - CONFIG['phi2_max']
    return df

# Morse * angular modulation
def morse_with_angles(vars, De, re, a, C1, C2):
    r_val, phi1_val, phi2_val = vars
    radial = De * (np.exp(-2*a*(r_val - re)) - 2*np.exp(-a*(r_val - re)))
    angular = 1 + C1*np.cos(np.radians(phi2_val - phi1_val)) \
                + C2*np.cos(2*np.radians(phi2_val - phi1_val))
    return radial * angular

def oriented_lj(vars, epsilon, sigma, a, m, alpha):
    r_val, phi1_val, phi2_val = vars
    radial = epsilon * 4 * ((sigma/(r_val))^12 - (sigma/(r_val))^6)
    angular = 1 + a*np.cos(m * np.radians(phi2_val - phi1_val) + alpha)
    return radial + angular


df = process_data(data_path)

phi1 = df['phi1'].values
phi2 = df['phi2'].values
r    = df['r'].values
E    = df['energy'].values

# ------------------------
# Choose r slice for plotting
# ------------------------
r_target = 9.0
mask = np.isclose(r, r_target, atol=0.01)

df_slice = df[mask].copy()

# ------------------------
# Fit model (on slice or full dataset)
# ------------------------
grid_df = df_slice.pivot_table(index='phi2', columns='phi1', values='energy')
Phi1, Phi2 = np.meshgrid(grid_df.columns, grid_df.index)
E_dftb_grid = grid_df.values

fit_mask = ~mask     # mask sclice, ~mask fitting on full data

initial_guess = [0.5, 5.0, 1.0, 0.1, 0.05]
popt_morse, pcov = curve_fit(
    morse_with_angles,
    (r[fit_mask], phi1[fit_mask], phi2[fit_mask]),
    E[fit_mask],
    p0=initial_guess
)
E_model_grid_morse = morse_with_angles((r_target*np.ones_like(Phi1), Phi1, Phi2), *popt_morse)
print("Fitted parameters:", popt_morse)

# initial_guess_LJ = [1, 1, 0.001, 2, 3.14]
# popt_lj, pcov2 = curve_fit(
#     oriented_lj,
#     (r[fit_mask], phi1[fit_mask], phi2[fit_mask]),
#     E[fit_mask],
#     p0=initial_guess
# )
# E_model_grid_lj = morse_with_angles((r_target*np.ones_like(Phi1), Phi1, Phi2), *popt_lj)
# print("Fitted parameters:", popt_lj)

E_model_grid = E_model_grid_morse
# ------------------------
# Plot comparison
# ------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True, sharey=True)

c0 = axes[0].contourf(Phi1, Phi2, E_dftb_grid, levels=50, cmap='viridis')
axes[0].set_title(f"DFTB Energy (r={r_target})")
axes[0].set_xlabel("phi1 (deg)")
axes[0].set_ylabel("phi2 (deg)")
fig.colorbar(c0, ax=axes[0], shrink=1)

c1 = axes[1].contourf(Phi1, Phi2, E_model_grid, levels=50, cmap='viridis')
axes[1].set_title("Morse+Angles Model")
axes[1].set_xlabel("phi1 (deg)")
fig.colorbar(c1, ax=axes[1], shrink=1)

plt.tight_layout()
plt.show()
